# Занятие 8. Практика: анализ логов во времени и продуктовая витрина

На прошлом занятии разобрали теорию: типы **`Timestamp`** и **`Timedelta`**, преобразование
текста в даты через **`pd.to_datetime()`** и аксессор **`.dt`**, интервалы между событиями,
группировки **`.groupby().agg()`** и сводные таблицы **`pd.pivot_table()`**.

Сегодня применяем это к логам интернет-магазина. Сначала **коротко вспомним инструменты**
на знакомом `orders.csv`

**План:**
1. повторение инструментов (на `orders.csv`);
2. Практика на данных `shop_orders.csv`.


In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 220)
pd.set_option('display.precision', 2)

---
## Повторение инструментов

Вспомним приёмы на знакомом `orders.csv` (заказы из прошлого урока).

In [ ]:
df = pd.read_csv('data/orders.csv', sep=';')
print('orders.csv:', df.shape)

### `pd.to_datetime()` и `.dt`

Текстовую дату превращаем в `Timestamp`, из него достаём компоненты через `.dt`:

In [ ]:
ts = pd.to_datetime('2024-03-15 14:30')
print('год:', ts.year, '| час:', ts.hour, '| день недели:', ts.day_name())

**Вывод:** у столбца-даты так же работают `.dt.year`, `.dt.hour`, `.dt.dayofweek`, `.dt.day_name()` — сразу для всей колонки.

### Интервалы между событиями

Разность двух дат — это `Timedelta`. Число дней получаем делением секунд на 86400:

In [ ]:
b = pd.Timestamp('2024-05-01 12:30')
a = pd.Timestamp('2024-05-05 09:00')
print(a - b, '=', round((a - b).total_seconds() / 86400, 2), 'дня')

**Вывод:** `.dt.total_seconds() / 86400` переводит интервал в дни с дробной частью.

### `groupby().agg()` — сводка по группам

Именованная агрегация: каждому столбцу отчёта — своя функция:

In [ ]:
df.groupby('payment').agg(
    заказов=('order_id', 'count'),
    средний_чек=('amount', 'mean'),
).round(1)

**Вывод:** одним вызовом собрали число заказов и средний чек по каждому способу оплаты.

### `pd.pivot_table()` — двумерный отчёт

Одна переменная по строкам, другая по столбцам, в клетках — агрегат:

In [ ]:
pd.pivot_table(df, index='city', columns='payment',
               values='order_id', aggfunc='count').fillna(0).astype(int).head()

**Вывод:** получили таблицу «город × способ оплаты» с числом заказов в клетках.

---
## Практика

Ниже — **12 заданий** (максимум **30 баллов**). В каждом задании:
- **напишите код** в ячейке `# Ваш код` — код обязателен, ответ должен считаться им;
- **выведите результат** через `print()` или последней строкой.

Во многих заданиях есть пункт **Вывод:** — там нужно короткое предложение с трактовкой
результата (что получилось и что это значит).

### Датасет практики — `shop_orders.csv`

**1365 заказов** интернет-магазина, **15 колонок**:

| Столбец | Что это |
|---|---|
| `order_id` | номер заказа |
| `created_at` | дата и время оформления |
| `delivered_at` | дата и время доставки (у части пусто) |
| `customer_id` | клиент |
| `segment` | сегмент клиента: `New` / `Returning` / `VIP` |
| `channel` | канал привлечения: Ads / Organic / Email / Social / Referral |
| `device` | устройство: Mobile / Desktop / Tablet |
| `city` | город |
| `category` | категория товара |
| `payment` | способ оплаты |
| `amount` | сумма заказа, руб. |
| `quantity` | число позиций |
| `discount_pct` | скидка, % |
| `delivery_days` | срок доставки, дней (пусто, если не доставлен) |
| `rating` | оценка 1–5 (пусто, если без отзыва) |

Загружаем его и дальше работаем **с ним**:

In [ ]:
df = pd.read_csv('data/shop_orders.csv', sep=';')
print('shop_orders.csv:', df.shape)
df.head()

### <font color='DarkOrange'>Задание 1 [1 балл]</font>

Столбцы `created_at` и `delivered_at` пока текстовые. Преобразуйте **оба** в тип даты-времени через `pd.to_datetime()` и выведите их типы.

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 2 [2 балла]</font>

Из `created_at` извлеките два признака в новые столбцы: `час` (`.dt.hour`) и `день_недели` (`.dt.day_name()`). Выведите первые строки этих столбцов.

In [ ]:
# Ваш код


### <font color='DarkOrange'>Задание 3 [2 балла]</font>

В какой **час суток** заказов больше всего? Найдите пиковый час и число заказов в нём (`value_counts`).

In [ ]:
# Ваш код


**Вывод:** *одним предложением — на какое время суток приходится пик и с чем это может быть связано.*

### <font color='DarkOrange'>Задание 4 [3 балла]</font>

Постройте **матрицу активности по устройствам**: `pivot_table` с устройством (`device`) по строкам, часом (`час`) по столбцам и числом заказов в клетках (`aggfunc='count'`). По матрице видно, в какие часы заказывают с разных устройств.

In [ ]:
# Ваш код


**Вывод:** *одним предложением — какое устройство используют днём, а какие вечером (видно по тому, в каких часах стоят числа).*

### <font color='DarkOrange'>Задание 5 [2 балла]</font>

Посчитайте **средний чек** (`amount`) по сегментам клиентов (`segment`). Отсортируйте по возрастанию.

In [ ]:
# Ваш код


**Вывод:** *одним предложением — насколько VIP дороже New и что это говорит о ценности сегмента.*

### <font color='DarkOrange'>Задание 6 [4 балла]</font>

Посчитайте **средний интервал между покупками одного клиента в разрезе сегмента**. Отсортируйте по `customer_id` и `created_at`, возьмите `.groupby('customer_id')['created_at'].diff()`, переведите в дни, затем усредните по `segment`.

In [ ]:
# Ваш код


**Вывод:** *одним предложением — какой сегмент возвращается за покупками чаще всего и во сколько раз.*

### <font color='DarkOrange'>Задание 7 [3 балла]</font>

Соберите **продуктовую витрину** по категориям через `groupby('category').agg(...)`: `заказов` (count), `средний_чек` (mean `amount`), `выручка` (sum `amount`), `клиентов` (nunique `customer_id`). Отсортируйте по выручке по убыванию.

In [ ]:
# Ваш код


**Вывод:** *одним предложением — какая категория приносит больше всего выручки и за счёт чего (объём заказов или высокий чек).*

### <font color='DarkOrange'>Задание 8 [3 балла]</font>

Найдите **самый активный день недели** — в какой день заказов больше всего. На выбор: постройте `pivot_table` день недели × час и просуммируйте по строкам, **или** сгруппируйте по `день_недели` и посчитайте заказы.

In [ ]:
# Ваш код


**Вывод:** *одним предложением — есть ли выраженный «главный» день недели или заказы распределены ровно.*

### <font color='DarkOrange'>Задание 9 [3 балла]</font>

Посчитайте **выручку по каналам** (`channel`, сумма `amount`) и отдельно — **долю выручки от VIP** в каждом канале (выручка VIP в канале / вся выручка канала). Какой канал приводит самых ценных клиентов?

In [ ]:
# Ваш код


**Вывод:** *одним предложением — какой канал одновременно самый крупный по выручке и приводит больше всего денег от VIP.*

### <font color='DarkOrange'>Задание 10 [2 балла]</font>

Оцените **эффективность скидок**. Создайте признак `со_скидкой` (`discount_pct > 0`) и сравните для заказов со скидкой и без: средний чек (`amount`), среднюю корзину (`quantity`) и число заказов.

In [ ]:
# Ваш код


**Вывод:** *одним предложением — помогают ли скидки продавать больше (растёт ли корзина) или просто снижают чек.*

### <font color='DarkOrange'>Задание 11 [3 балла]</font>

Разбейте сутки на **4 части** и сравните их. Создайте столбец `время_суток` из `час`: ночь (0–5), утро (6–11), день (12–17), вечер (18–23). Посчитайте по каждой части число заказов и средний чек (`groupby().agg()`). Когда заказов больше всего, а когда средний чек выше?

In [ ]:
# Ваш код


**Вывод:** *одним предложением — совпадает ли время, когда заказов больше всего, со временем самого высокого среднего чека.*

### <font color='DarkOrange'>Задание 12 [2 балла]</font>

Аналитика. Посчитайте **выручку на одного клиента по сегментам** (сумма `amount` / число уникальных клиентов). Опираясь на это и выводы заданий 5, 6 и 9, предложите, на какой сегмент и через какой канал выгоднее тратить маркетинг. Обоснуйте в выводе.

In [ ]:
# Ваш код


**Вывод:** *2–3 предложения — на какой сегмент и канал делать ставку и почему (свяжите ценность клиента, частоту покупок и канал привлечения VIP).*